<a href="https://colab.research.google.com/github/iamajeet/colab-workbook/blob/main/agentic_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import requests
# ============================================
# API KEY: Get your key from https://aicafe.hcl.com/AICafe/#/tutorials/api-docs
# IMPORTANT: Enter your API key in the GUI instead of hardcoding it here
# ============================================
API_KEY = "d1abf648-8075-4df2-b431-0549d6d8866e"  # <-- Use the GUI to enter your key
# --- Endpoint config ---
API_URL = (
    "https://aicafe.hcl.com/AICafeService/api/v1/subscription/openai/"
    "deployments/gpt-4.1/chat/completions?api-version=2024-12-01-preview"
)
SYSTEM_PROMPT = """
You are a helpful AI agent.
- Answer clearly and concisely.
- Ask for clarification only when truly needed.
"""
def run_agent():
    print("Simple AI agent. Type 'quit' to exit.\n")
    while True:
        user_input = input("You: ").strip()
        if user_input.lower() in {"quit", "exit"}:
            print("Agent: Goodbye! 👋")
            break
        # Build request body (OpenAI-style)
        body = {
            "model": "gpt-4.1",
            "messages": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_input},
            ],
            "temperature": 0.7,
        }
        headers = {
            "api-key": API_KEY,
            "Content-Type": "application/json",
            "Accept": "application/json",
        }
        try:
            response = requests.post(API_URL, headers=headers, json=body, timeout=60)
            response.raise_for_status()
            data = response.json()
            agent_reply = data["choices"][0]["message"]["content"].strip()
            print(f"Agent: {agent_reply}\n")
        except Exception as e:
            print(f"Error: {e}")
if __name__ == "__main__":
    run_agent()


In [ ]:
!pip install langchain langchain-community langchain-google-genai wikipedia


In [ ]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_community.tools import WikipediaQueryRun

In [ ]:
from google.colab import userdata
from google import genai
# Load API key from Colab Secrets into environment variable
os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

In [ ]:
# Set your Gemini API key
#os.environ["GOOGLE_API_KEY"] = "your_gemini_api_key"

# Create Gemini LLM
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)

# Create Wikipedia tool
wiki = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())

# Ask user question
query = input("Ask a question: ")

# Retrieve Wikipedia content
wiki_result = wiki.run(query)
print("\nWikipedia result:")
print(wiki_result)

# Send to Gemini for summarization
prompt = f"""
Answer the question using the information from Wikipedia.

Question: {query}

Wikipedia information:
{wiki_result}

Provide a clear and concise answer.
"""

Ask a question: who is sania mirza

Wikipedia result:
Page: Sania Mirza
Summary: Sania Mirza ([ˈsaːnijaː ˈmirzaː]; born 15 November 1986) is an Indian former professional tennis player. A former doubles world No. 1, she won six major titles – three in women's doubles and three in mixed doubles. From 2003 until her retirement from singles in 2013, she was ranked by the Women's Tennis Association as the No. 1 Indian in singles. Throughout her career, Mirza has established herself as one of the most known, highest-paid, and influential athletes in India.

In singles, Mirza had wins over Svetlana Kuznetsova, Vera Zvonareva, and Marion Bartoli, as well as former world-number-ones Martina Hingis, Dinara Safina, and Victoria Azarenka. She is the highest-ranked Indian female player ever, peaking at world No. 27 in mid-2007. However, a major wrist injury caused her to shift to doubles. Mirza has achieved a number of firsts for women's tennis in India, including reaching the one million-US$ mark

In [ ]:

response = llm.invoke(prompt)

print("\nAnswer:")
print(response.content)


Answer:
Sania Mirza is an Indian former professional tennis player, born on 15 November 1986. She was a former doubles world No. 1 and won six major titles (three in women's doubles and three in mixed doubles). Throughout her career, she established herself as one of the most known, highest-paid, and influential athletes in India. Mirza retired from professional tennis in February 2023.


In [ ]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_community.tools import WikipediaQueryRun
from langchain_core.messages import HumanMessage


import os
from google.colab import userdata
from google import genai
# Load API key from Colab Secrets into environment variable
os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

In [ ]:
# Set Gemini API Key
# os.environ["GOOGLE_API_KEY"] = "your_gemini_api_key"

# Create Gemini LLM
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)

# Wikipedia tool
wiki_api = WikipediaAPIWrapper()
wiki_tool = WikipediaQueryRun(api_wrapper=wiki_api)

# Function to answer questions
def ask_agent(question):

    # Step 1: search wikipedia
    wiki_result = wiki_tool.run(question)

    # Step 2: send to Gemini
    prompt = f"""
    Answer the user's question using the Wikipedia information below.

    Question:
    {question}

    Wikipedia information:
    {wiki_result}

    Give a clear and short answer.
    """

    response = llm.invoke([HumanMessage(content=prompt)])

    return response.content



In [ ]:
# Interactive loop
while True:
    query = input("\nAsk a question (type exit to stop): ")

    if query.lower() == "exit":
        break

    answer = ask_agent(query)

    print("\nAnswer:", answer)


Ask a question (type exit to stop): who is john abraham

Answer: John Abraham is an Indian actor, writer, and film producer who primarily works in Hindi films. He was also a former model.

Ask a question (type exit to stop): exit


In [ ]:

import os
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_community.tools import WikipediaQueryRun
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableMap, RunnableLambda

In [ ]:
import os
from google.colab import userdata
from google import genai
# Load API key from Colab Secrets into environment variable
os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

In [ ]:
# Set API key
# os.environ["GOOGLE_API_KEY"] = "your_google_api_key"

# Gemini LLM
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)

# Wikipedia tool
wiki_tool = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())

# Step 1 — Retrieve context
retriever = RunnableLambda(lambda topic: wiki_tool.run(topic))

# Step 2 — Summarization
summary_prompt = ChatPromptTemplate.from_template(
"""
Summarize the following information in 5 sentences.

Information:
{context}
"""
)

summary_chain = summary_prompt | llm

# Step 3 — Quiz generation
quiz_prompt = ChatPromptTemplate.from_template(
"""
Based on the summary below, generate 3 quiz questions.

Summary:
{summary}
"""
)

quiz_chain = quiz_prompt | llm


In [ ]:
# Full pipeline
pipeline = (
    RunnableMap({"context": retriever})
    | summary_chain
    | RunnableLambda(lambda msg: {"summary": msg.content})
    | quiz_chain
)

# Run pipeline
topic = input("Enter a topic: ")

result = pipeline.invoke(topic)

print("\nQuiz Questions:\n")
print(result.content)

Enter a topic: cricket

Quiz Questions:

Here are 3 quiz questions based on the summary:

1.  How many players are on each team in a game of cricket?
2.  Name two ways a batter can score runs in cricket.
3.  Which country has won the most ICC Men's Cricket World Cups, and how many have they won?


In [ ]:
!pip install -U langchain langchain-community langchain-core langchain-google-genai

In [ ]:

import os

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory

In [ ]:
# ----------------------------
# Configure Gemini API
# ----------------------------

# os.environ["GOOGLE_API_KEY"] = "YOUR_GOOGLE_API_KEY"
# genai.configure(api_key=os.environ["GOOGLE_API_KEY"])
import os
from google.colab import userdata
from google import genai
# Load API key from Colab Secrets into environment variable
os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

In [ ]:
# ---------------------------
# LLM
# ---------------------------

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)


# ---------------------------
# Tool
# ---------------------------

@tool
def calculator(expression: str) -> str:
    """Evaluate a math expression."""
    return str(eval(expression))


tools = [calculator]


# ---------------------------
# Prompt
# ---------------------------

prompt = ChatPromptTemplate.from_messages(
[
("system", "You are a helpful assistant. Use the calculator tool when needed."),
("placeholder", "{chat_history}"),
("human", "{input}")
]
)


# ---------------------------
# Tool binding
# ---------------------------

llm_with_tools = llm.bind_tools(tools)



In [ ]:
# ---------------------------
# Chain
# ---------------------------

chain = prompt | llm_with_tools


# ---------------------------
# Memory
# ---------------------------

store = {}

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]


agent = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history"
)


In [ ]:

# ---------------------------
# Chat loop
# ---------------------------

while True:

    query = input("\nYou: ")

    if query.lower() == "exit":
        break

    response = agent.invoke(
        {"input": query},
        config={"configurable": {"session_id": "demo"}}
    )

    print("\nAgent:", response.content)


In [ ]:
# Document chunking
# Persistent vector DB (FAISS saved to disk)
# LangChain retriever
# LCEL pipeline (|)
# Gemini 2.5 Flash answering

In [1]:
!pip install langchain langchain-community langchain-google-genai faiss-cpu sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 31.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 67.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 48.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.5/503.5 kB 35.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.1 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.18
    Uninstalling langchain-core-1.2.18:
      Successfully uninstalled langchain-core-1.2.18
ERROR: pip's dependency resolver does not currently take into account all the packages that are instal

In [2]:
import os
import google.generativeai as genai

from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

from langchain_google_genai import ChatGoogleGenerativeAI

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [3]:
# -----------------------------
# 1. Configure Gemini API
# -----------------------------

#os.environ["GOOGLE_API_KEY"] = "YOUR_GOOGLE_API_KEY"
#genai.configure(api_key=os.environ["GOOGLE_API_KEY"])

import os
from google.colab import userdata
from google import genai
# Load API key from Colab Secrets into environment variable
os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

In [4]:
# -----------------------------
# 2. Sample Documents
# -----------------------------

documents = [
    Document(page_content="""Tesla produces electric cars and energy storage products.
    The company was founded by Elon Musk and focuses on sustainable transportation."""),

    Document(page_content="""Toyota is a Japanese automobile manufacturer known for
    hybrid vehicles such as the Prius. The company pioneered hybrid technology."""),

    Document(page_content="""BMW is a German manufacturer of luxury vehicles and
    motorcycles. It is known for performance and premium engineering.""")
]


In [12]:
# -----------------------------
# 3. Document Chunking
# -----------------------------

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=50
)

docs = text_splitter.split_documents(documents)

In [13]:
# -----------------------------
# 4. Embedding Model
# -----------------------------

embedding_model = HuggingFaceEmbeddings(
   model_name="sentence-transformers/all-MiniLM-L6-v2"
)

/tmp/ipykernel_640/1507230227.py:5: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:

# client = genai.Client()

# embedding_model = client.models.embed_content(
#         model="gemini-embedding-001",
#         contents=docs
# )


In [14]:


# -----------------------------
# 5. Persistent Vector Database
# -----------------------------

db_path = "vector_db"

if os.path.exists(db_path):

    vector_db = FAISS.load_local(
        db_path,
        embedding_model,
        allow_dangerous_deserialization=True
    )

else:

    vector_db = FAISS.from_documents(docs, embedding_model)

    vector_db.save_local(db_path)


In [15]:
# -----------------------------
# 6. Create Retriever
# -----------------------------

retriever = vector_db.as_retriever(
    search_kwargs={"k": 2}
)


# -----------------------------
# 7. Create Gemini LLM
# -----------------------------

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)


In [16]:
# -----------------------------
# 8. Prompt Template
# -----------------------------

prompt = ChatPromptTemplate.from_template(
"""
You are an AI assistant answering questions using retrieved context.

Context:
{context}

Question:
{question}

Answer the question clearly using only the provided context.
"""
)


# -----------------------------
# 9. Helper Function
# -----------------------------

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


# -----------------------------
# 10. LCEL RAG Pipeline
# -----------------------------

rag_chain = (
    {
        "context": retriever | RunnableLambda(format_docs),
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
)

In [17]:
# -----------------------------
# 11. Interactive Query Loop
# -----------------------------

while True:

    question = input("\nAsk a question (type 'exit' to stop): ")

    if question.lower() == "exit":
        break

    response = rag_chain.invoke(question)

    print("\nAnswer:\n")
    print(response.content)


Ask a question (type 'exit' to stop): what is toyota?

Answer:

Toyota is a Japanese automobile manufacturer known for hybrid vehicles such as the Prius. The company pioneered hybrid technology.

Ask a question (type 'exit' to stop): toyota headquarter?

Answer:

The provided context states that Toyota is a Japanese automobile manufacturer, but it does not specify the location of its headquarters.

Ask a question (type 'exit' to stop): toyota from?

Answer:

Toyota is from Japan.

Ask a question (type 'exit' to stop): exit
